In [0]:
import re
from pyspark.sql.functions import *
from pyspark.sql import Window
path = '/Volumes/ai_dev/input/ai_dataset/10000Records.csv'
df = spark.read.format('csv').options(header='true',inferSchema='true').load(path)

# fetch highest salary employ
# res = df.orderBy(col('Salary').desc()).limit(1)
res = df.orderBy(col('Salary').desc())

res = res.select('First Name','state','Salary')
# display(res)
res.show(2)

+----------+-----+------+
|First Name|state|Salary|
+----------+-----+------+
|     Kenna|   AL|199971|
|      Dick|   KY|199969|
+----------+-----+------+
only showing top 2 rows


In [0]:
import re
from pyspark.sql.functions import *
from pyspark.sql import Window
path = '/Volumes/ai_dev/input/ai_dataset/10000Records.csv'
df = spark.read.format('csv').options(header='true',inferSchema='true').load(path)

## fetch 3rd highest salary employ state wise (i.e in each state)
win1 = Window.partitionBy(col('state')).orderBy(col('Salary').desc())
res = df.withColumn('rnk',rank().over(win1)).\
    withColumn('drnk',dense_rank().over(win1)).\
        withColumn('rno',row_number().over(win1))

res = res.select('First Name','state','Salary','drnk').filter(col('drnk')==3).drop('drnk')
display(res)

First Name,state,Salary
Donn,AK,197758
Fred,AL,199440
Wendy,AR,194178
Stephnie,AZ,194090
Conrad,CA,199028
Ike,CO,194985
Ruby,CT,194162
Taryn,DC,189132
Dolly,DE,178599
Lynwood,FL,198261


In [0]:
df.createOrReplaceTempView('employ')
# fetch 3rd highest salary employ state wise (i.e in each state)
res = spark.sql("""
                select * from (
          select `First Name`,state,Salary,dense_rank() over(partition by state order by Salary desc) as rn from employ 
                ) where rn=3
          """)
display(res)          

First Name,state,Salary,rn
Donn,AK,197758,3
Fred,AL,199440,3
Wendy,AR,194178,3
Stephnie,AZ,194090,3
Conrad,CA,199028,3
Ike,CO,194985,3
Ruby,CT,194162,3
Taryn,DC,189132,3
Dolly,DE,178599,3
Lynwood,FL,198261,3


In [0]:
import re
from pyspark.sql.functions import *
from pyspark.sql import Window
path = '/Volumes/ai_dev/input/ai_dataset/10000Records.csv'
df = spark.read.format('csv').options(header='true',inferSchema='true').load(path)

# lead() & lag()
# win2= Window.orderBy(col('Salary'))
# res = df.withColumn('prev_sal',lag(col('Salary')).over(win2)).\
#     withColumn('next_sal',lead(col('Salary')).over(win2))

# first() & last()
win3 = Window.partitionBy(col('state'))
res = df.withColumn('first_sal',first(col('Salary')).over(win3)).\
    withColumn('last_sal',last(col('Salary')).over(win3))

res = res.select('First Name','state','Salary','first_sal','last_sal')

display(res)

First Name,state,Salary,first_sal,last_sal
Aurora,AK,141342,141342,40204
Lovetta,AK,67952,141342,40204
Alexis,AK,193600,141342,40204
Reyes,AK,115893,141342,40204
Renea,AK,154043,141342,40204
Grant,AK,67534,141342,40204
Sebastian,AK,152499,141342,40204
Kanisha,AK,128262,141342,40204
Sueann,AK,101287,141342,40204
Mee,AK,158746,141342,40204


In [0]:
res = spark.sql("""
          select `First Name`,state,Salary, lead(Salary) over(order by salary ) as next_salary,
          lag(salary) over(order by salary) as prev_salary from employ 
          """)
display(res) 

First Name,state,Salary,next_salary,prev_salary
Royce,NY,40007,40009,null
Jodi,CA,40009,40009,40007
Ben,CO,40009,40010,40009
Peter,GA,40010,40015,40009
Lorina,OH,40015,40017,40010
Reginald,MN,40017,40021,40015
Ben,AK,40021,40049,40017
Anja,VA,40049,40051,40021
Sharan,MD,40051,40061,40049
Santo,FL,40061,40069,40051


year,revenue
2017,12000.0
2018,13500.5
2019,14200.0
2020,15500.2
2021,16000.0
2022,17000.8
2023,18500.0
2024,19000.3
2025,21000.0
2026,22000.7


In [0]:
%sql
select * from employ

In [0]:
data = [(2017, 12000), (2018, 15000), (2019, 18000), (2020, 21000), (2021, 25000),
        (2022, 30000), (2023, 35000), (2024, 40000), (2025, 45000), (2026, 50000)]
columns = ['year', 'revenue']
df_revenue = spark.createDataFrame(data, columns)

res = df_revenue.withColumn('prev_revenue',lag(col('revenue')).over(Window.orderBy(col('year'))))

res = res.withColumn('YOY_Growth',round((col('revenue')-col('prev_revenue'))/col('revenue')*100,2))
display(res)
# display(df_revenue)

## YOY growth

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


year,revenue,prev_revenue,YOY_Growth
2017,12000,null,null
2018,15000,12000,20.0
2019,18000,15000,16.67
2020,21000,18000,14.29
2021,25000,21000,16.0
2022,30000,25000,16.67
2023,35000,30000,14.29
2024,40000,35000,12.5
2025,45000,40000,11.11
2026,50000,45000,10.0


In [0]:
2020- 12000
2021 -24000

15000-12000/15000


In [0]:
(24000-12000)/12000 * 100

100.0

In [0]:
path = '/Volumes/student_demo/standard/input_data/employees.csv'

df = spark.read.format('csv').options(header='true',inferSchema='true').load(path)

df1 = df.withColumn('MANAGER_ID',regexp_replace(col('MANAGER_ID'),"-","0"))

# type casting
new_df = df1.withColumn('HIRE_DATE',to_date(col('HIRE_DATE'),'dd-MMM-yy')).\
    withColumn('MANAGER_ID',col('MANAGER_ID').cast('integer')).drop('COMMISSION_PCT')

# calculate average salary & replace null salary with calculated avaerage salary
average_salary = new_df.agg(round(avg(col('SALARY')),0)).first()[0]
# print(average_salary) -->6260

## handle null values
# fillna() --> use to handle null values

# res = new_df.fillna(0)
res = new_df.fillna({'LAST_NAME':'UNKNOWN','HIRE_DATE':'1990-01-01','SALARY': average_salary})

display(res)

# new_df.printSchema()
# 1990-01-01

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,MANAGER_ID,DEPARTMENT_ID
198,Donald,OConnell,DOCONNEL,650.507.9833,2007-06-21,SH_CLERK,2600,124,50
199,Douglas,Grant,DGRANT,650.507.9844,2008-01-13,SH_CLERK,2600,124,50
200,Jennifer,Whalen,JWHALEN,515.123.4444,2003-09-17,AD_ASST,4400,101,10
200,Jennifer,Whalen,JWHALEN,515.123.4444,2003-09-17,AD_ASST,4400,101,10
201,Michael,Hartstein,MHARTSTE,515.123.5555,2004-02-17,MK_MAN,13000,100,20
202,Pat,UNKNOWN,PFAY,603.123.6666,2005-08-17,MK_REP,6000,201,20
203,Susan,Mavris,SMAVRIS,515.123.7777,2002-06-07,HR_REP,6260,101,40
205,Shelley,Higgins,SHIGGINS,515.123.8080,2002-06-07,AC_MGR,12008,101,110
204,Hermann,Baer,HBAER,515.123.8888,1990-01-01,PR_REP,10000,101,70
205,Shelley,Higgins,SHIGGINS,515.123.8080,2002-06-07,AC_MGR,12008,101,110
